# Global Feature Selection

## Purpose
Select a global feature set for each experiment setting (`ds`, `ds_desc`, `ds_md`, `ds_desc_md`) from pose-only rows.

## Input
- CSV table with docking scores, descriptors, MD features, RMSD, and labels (output of the RMSD/labeling step)

## Output
- Feature list CSVs (one per experiment setting) in the `features_global/` output directory
- Preprocessing logs in the `features_global_logs/` output directory

## Run Order
1. MD/descriptors/RMSD labeling steps (notebooks 01--03)
2. This notebook
3. LOPOCV LightGBM optimization notebook

In [ ]:
# ================================================================
# Cell 1: Configuration
# ================================================================

# =====================
# Feature selection (global)
#  - From pose rows only: confirm and save feature set after preprocessing
#    (constant/missing/duplicate/NZV/high-correlation removal)
#  - Build seed-independent "features_global" (strategy B)
# =====================

# Update INPUT_CSV and OUT_DIR to match your local directory structure before running.
INPUT_CSV = "/path/to/input.csv"
OUT_DIR   = "/path/to/output"

EXPERIMENTS = [
    {"name": "ds",         "use_desc": False, "use_md": False},
    {"name": "ds_desc",    "use_desc": True,  "use_md": False},
    {"name": "ds_md",      "use_desc": False, "use_md": True},
    {"name": "ds_desc_md", "use_desc": True,  "use_md": True},
]

drop_cols_base = [
    "pdb_id", "ligand_id", "smiles", "pocket_id",
    "label_2p5", "label_5p0", "rmsd_gninaA", "pdb_offset",
    "pocket_score_site1", "pocket_score_site2", "pocket_score_site3",
    "pocket_score_max",
]

NZV_UNIQUE_RATIO_MAX = 0.10
NZV_FREQ_CUT         = 19
HIGH_CORR_ABS_THRES  = 0.90

DS_COLS = ["vina_affinity", "cnn_pose_score", "cnn_affinity"]


In [ ]:
# ================================================================
# Cell 2: Libraries
# ================================================================

import os
from pathlib import Path
from typing import Dict, Tuple, List
import numpy as np
import pandas as pd


In [ ]:
# ================================================================
# Cell 3: Data loading
# ================================================================

# ---- Data ----
df_raw = pd.read_csv(INPUT_CSV)
df_raw = df_raw[df_raw["ligand_id"].astype(str).str.startswith("pose")].copy()
print(f"[Data] rows (pose only): {len(df_raw)}")


In [ ]:
# ================================================================
# Cell 4: Feature-selection helpers
# ================================================================

# ---- Preprocessing functions (identical to training notebook) ----
def build_feature_matrix(df_in: pd.DataFrame, use_desc: bool, use_md: bool) -> Tuple[pd.DataFrame, Dict]:
    drop_cols = drop_cols_base.copy()
    num_cols_all = df_in.select_dtypes(include=[np.number]).columns.tolist()

    md_all   = [c for c in df_in.columns if c.startswith("md_")]
    ds_all   = [c for c in DS_COLS if c in df_in.columns]
    protect = set(drop_cols + ds_all + md_all)
    desc_pool = [c for c in num_cols_all if c not in protect]

    use_cols: List[str] = []
    use_cols += ds_all
    if use_desc: use_cols += desc_pool
    if use_md:   use_cols += [c for c in md_all if c in num_cols_all]
    use_cols = list(dict.fromkeys(use_cols))

    X0 = df_in[use_cols].copy()

    # (A) Constant columns
    const_cols = [c for c in X0.columns if X0[c].nunique(dropna=False) <= 1]
    X1 = X0.drop(columns=const_cols)

    # (B) Columns with missing values
    missing_cols = [c for c in X1.columns if X1[c].isna().any()]
    X2 = X1.drop(columns=missing_cols, errors="ignore")

    # (C) Fully duplicated columns
    if X2.shape[1] > 0:
        keep = X2.T.drop_duplicates().index.tolist()
        dup_cols = list(set(X2.columns) - set(keep))
        X3 = X2.drop(columns=dup_cols)
    else:
        dup_cols = []
        X3 = X2

    # (D) NZV
    nzv_cols = []
    n = len(X3)
    for c in X3.columns:
        vc = X3[c].value_counts(dropna=False)
        unique_ratio = (vc.size / n) if n > 0 else 0.0
        freq_ratio = vc.iloc[0] / vc.iloc[1] if len(vc) > 1 else np.inf
        if (unique_ratio <= NZV_UNIQUE_RATIO_MAX) and (freq_ratio > NZV_FREQ_CUT):
            nzv_cols.append(c)
    X4 = X3.drop(columns=nzv_cols, errors="ignore")

    # (E) High correlation
    high_corr_cols = []
    if X4.shape[1] >= 2:
        corr = X4.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        high_corr_cols = [col for col in upper.columns if any(upper[col] > HIGH_CORR_ABS_THRES)]
        X5 = X4.drop(columns=high_corr_cols, errors="ignore")
    else:
        X5 = X4

    log = {
        "use_desc": use_desc,
        "use_md": use_md,
        "use_cols_initial": use_cols,
        "const_cols": const_cols,
        "missing_cols": missing_cols,
        "dup_cols": dup_cols,
        "nzv_cols": nzv_cols,
        "high_corr_cols": high_corr_cols,
        "final_cols": X5.columns.tolist(),
    }
    return X5, log


In [ ]:
# ================================================================
# Cell 5: Save feature lists
# ================================================================

# ---- Output directories ----
feat_dir = Path(OUT_DIR) / "features_global"
feat_dir.mkdir(parents=True, exist_ok=True)

log_dir = Path(OUT_DIR) / "features_global_logs"
log_dir.mkdir(parents=True, exist_ok=True)

# ---- Run ----
for exp in EXPERIMENTS:
    name = exp["name"]
    X, log = build_feature_matrix(df_raw, use_desc=exp["use_desc"], use_md=exp["use_md"])

    out_csv = feat_dir / f"{name}.csv"
    pd.Series(X.columns, name="feature").to_csv(out_csv, index=False)

    # Log (what was removed)
    out_log = log_dir / f"{name}_log.json"
    import json
    with open(out_log, "w") as f:
        json.dump(log, f, indent=2)

    print(f"[OK] {name}: n_features={X.shape[1]}  saved -> {out_csv}")
